# 1. PA3 Starter: Geometry in Motion

This notebook walks through normalized DLT, finite-population RANSAC, a separate coarse-to-fine
translation study, and a planar-motion application. Plan for 10+ hours: about 1.5 to 2 hours on
Part A, 2.5 to 3 on Part B, 2.5 to 3 on Part C, and 1.5 to 2 on Part D, plus time for setup and
debugging. Save generated files only in `artifacts/`.

By the end you should be able to:

- derive DLT constraints for a stated vectorization convention and solve them with SVD,
- implement and debug finite-population RANSAC,
- derive and solve the Lucas-Kanade brightness-constancy system,
- tell geometric error apart from photometric error and visual error,
- state the assumptions behind homographies and differential alignment,
- show where your method works and where it fails.

The [assignment instructions (PDF)](assignment_instructions.pdf) have every question, requirement, and grading rule. This notebook holds the code you run for each part.


## 2. Integrity and permitted tools

Generative AI is not permitted for Part A or for the required Part B code in `student/pa3.py`. It is permitted in Parts C, D, and E if you disclose it and understand what it produced. Supplied cells are labeled `SUPPLIED`. Code you must write is marked `TODO(student)`.

In a fresh Colab runtime, run the next cell and upload the repository ZIP downloaded from GitHub when prompted. The cell extracts the repository and installs the Colab requirements. The notebook file alone is not enough.

In [ ]:
# SUPPLIED: locate a complete repository or upload its ZIP in Colab.
from pathlib import Path
import io
import os
import subprocess
import sys
import zipfile

REPOSITORY_MARKERS = (
    "requirements-colab.txt",
    "student/pa3.py",
    "common/pa3_utils.py",
    "data/setup_data.py",
)
KNOWN_EXTRACT_ROOT = Path(os.environ.get(
    "CV16720_REPOSITORY_EXTRACT_ROOT", "/content/cv16720_pa3_repository"
)).resolve()

def _is_repository_root(candidate):
    return all((candidate / marker).is_file() for marker in REPOSITORY_MARKERS)

def _single_repository_beneath(base):
    if not base.is_dir():
        return None
    if _is_repository_root(base):
        return base.resolve()
    roots = sorted(
        path.resolve() for path in base.iterdir()
        if path.is_dir() and _is_repository_root(path)
    )
    if len(roots) > 1:
        raise RuntimeError(
            f"Expected at most one PA3 repository beneath {base}; "
            f"found {len(roots)}."
        )
    return roots[0] if roots else None

def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if _is_repository_root(candidate):
            return candidate
    return _single_repository_beneath(KNOWN_EXTRACT_ROOT)

ROOT = _find_repository_root()
if ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Could not find the PA3 repository. Extract the complete repository ZIP and "
            "run this notebook from its root."
        ) from exc

    print("Upload the complete PA3 repository ZIP downloaded from GitHub.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one repository ZIP, then rerun this cell.")

    extract_root = KNOWN_EXTRACT_ROOT
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            target = (extract_root / member_path).resolve()
            mode = (member.external_attr >> 16) & 0o170000
            if (member_path.is_absolute() or ".." in member_path.parts
                    or (target != extract_root and extract_root not in target.parents)
                    or mode == 0o120000):
                raise RuntimeError(f"Unsafe ZIP member rejected: {member.filename!r}")
        archive.extractall(extract_root)

    ROOT = _single_repository_beneath(extract_root)
    if ROOT is None:
        raise RuntimeError(
            "The uploaded ZIP did not contain a PA3 repository root."
        )

os.chdir(ROOT)
requirements = ROOT / "requirements-colab.txt"
if "google.colab" in sys.modules and os.environ.get("CV16720_PA3_COLAB_ROOT") != str(ROOT):
    if not requirements.is_file():
        raise RuntimeError(f"Missing {requirements}; upload the complete repository ZIP.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)
    ])
    os.environ["CV16720_PA3_COLAB_ROOT"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import time
import numpy as np
import matplotlib.pyplot as plt

ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 16720
rng = np.random.default_rng(SEED)

from common.pa3_utils import (
    corner_transfer_rmse, draw_correspondences, draw_projected_quad,
    inlier_precision_recall, make_homography_case, make_translation_pair,
    manual_corner_metrics, project_points, sift_correspondences,
)


In [ ]:
# SUPPLIED: Colab/local equivalent of: python data/setup_data.py --synthetic --b4
# The synthetic cases are deterministic and need no network. The Oxford wall sequence is
# checksum-verified and required for the graded B4 sequence row.
from data.setup_data import download_b4, generate_synthetic
DATA_ROOT = ROOT / 'data' / 'generated'
synthetic_assets = generate_synthetic(DATA_ROOT)
print(f'generated {len(synthetic_assets)} synthetic assets')
try:
    wall_assets = download_b4(DATA_ROOT)
    print(f'wall sequence ready: {len(wall_assets)} checksum-verified files')
except OSError as error:
    print(f'Could not download the wall sequence: {error}')
    print('B4 will run the synthetic smoke check instead, which earns no B4 sequence credit.')
    print('Rerun this cell once you are online, or run: python data/setup_data.py --b4')


`symmetric_transfer_error` comes from your `student/pa3.py`, not from the supplied helpers. The next cell lists the helper imports so you can trace them in B1.

In [ ]:
# SUPPLIED: imports kept explicit for B1 tracing
from common.pa3_utils import (
    bilinear_sample, gaussian_pyramid, image_corners, image_gradients,
    to_homogeneous, tracking_validity, warp_image, write_h264_video,
)
import student.pa3 as pa3
print('NumPy', np.__version__, '| root', ROOT)

## 3. Coordinate and metric conventions

- `xy` is a float64 `(N,2)` array with columns `(x,y)`. Images are indexed `image[y,x]`.
- `H_src_to_dst` maps source points to destination points.
- Warping inverse-samples destination pixels from the source.
- DLT vectorization is column-major and recovery uses `order='F'`.
- Symmetric transfer error is the mean of the forward and backward squared distances, in pixels squared.
- A RANSAC threshold in pixels is squared before it is compared with this error.

In [ ]:
# SUPPLIED B1 anchor: homogeneous rows, column-vector projection, inverse sampling
xy = np.array([[10.0, 20.0], [30.0, 40.0]], dtype=np.float64)  # (N,2), columns x,y
xy_h = to_homogeneous(xy)                                      # (N,3)
H_src_to_dst = np.array([[1., 0., 3.], [0., 1., -2.], [0., 0., 1.]])
projected_h_rows = (H_src_to_dst @ xy_h.T).T
tiny = np.arange(36, dtype=np.float64).reshape(6, 6)
warped, warp_valid = warp_image(tiny, H_src_to_dst, tiny.shape)
sample_values, sample_valid = bilinear_sample(tiny, np.array([[1.5, 2.0], [-1.0, 2.0]]))
print(xy.shape, xy_h.shape, projected_h_rows.shape, sample_values, sample_valid)

## 4. B1: Code-reading responses

Answer B1.1 to B1.3 and B1.5 in the LaTeX template before implementing B2. Answer B1.4 after implementing B2, referring to the two design-matrix rows in your own DLT code. Use the anchor cell above and the supplied helper source to trace array shapes and column order, the forward transform versus inverse sampling, sampled values versus the validity mask, how the A1 design rows match `order='F'`, and how pyramid level zero relates to full-resolution coordinates.

## 5. B2: Normalized DLT

Implement the three B2 functions in `student/pa3.py`, reload the module, and run the checks. Do not place an alternate implementation in this notebook.

In Colab, open the Files pane and locate `student/pa3.py` beneath the repository directory printed above. Double-click it, edit the marked regions, and save with Ctrl+S (Cmd+S on macOS). If you edit locally instead, upload the edited file into that same `student` folder, replacing the starter. Python caches imported modules, so run the reload cell below after every save, then rerun the cells that depend on your functions.

Download your edited `pa3.py` often. Colab runtime files are temporary, and a disconnected or reset runtime erases them. Saving the notebook to Drive does not save separate Python files. The backup cell at the end of this notebook bundles your code and figures. Download the notebook itself with **File → Download → Download .ipynb**.


In [ ]:
# SUPPLIED: reload after editing student/pa3.py
pa3 = importlib.reload(pa3)
square = np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
normalized, T = pa3.normalize_points(square)
print('centroid:', normalized[:, :2].mean(0))
print('mean radius:', np.linalg.norm(normalized[:, :2], axis=1).mean())
print('T:\n', T)

In [ ]:
# SUPPLIED: exact non-symmetric transpose trap
src = np.array([[20.,15.],[610.,20.],[600.,450.],[25.,460.]])
H_true = np.array([[1.03,.14,27.],[-.08,.96,19.],[2.2e-4,-1.4e-4,1.]])
dst, valid = project_points(H_true, src)
H_est = pa3.estimate_homography_dlt(src, dst)
print('corner RMSE:', corner_transfer_rmse(H_est, H_true, (640,480)))
print('symmetric errors:', pa3.symmetric_transfer_error(H_est, src, dst))

## 6. B3: Finite-population RANSAC

Implement `ransac_homography`, then run the next cell for all four outlier fractions and five seeds. Build a table with direct and RANSAC corner RMSE, precision, recall, runtime, and trials.

In [ ]:
# SUPPLIED: the required B3 protocol. Run it unchanged so every row is comparable.
rows = []
for fraction in (0.0, 0.2, 0.4, 0.6):
    for seed in range(16720, 16725):
        case = make_homography_case(seed=seed, outlier_fraction=fraction)
        start = time.perf_counter()
        H_direct = pa3.estimate_homography_dlt(case['src_xy'], case['dst_xy'])
        direct_ms = 1000 * (time.perf_counter() - start)
        start = time.perf_counter()
        H_robust, mask, info = pa3.ransac_homography(
            case['src_xy'], case['dst_xy'], 3.0, 0.99, np.random.default_rng(SEED))
        ransac_ms = 1000 * (time.perf_counter() - start)
        metrics = inlier_precision_recall(mask, case['true_inliers'])
        rows.append(dict(fraction=fraction, seed=seed,
            direct_rmse=corner_transfer_rmse(H_direct, case['H_true'], case['image_size']),
            ransac_rmse=corner_transfer_rmse(H_robust, case['H_true'], case['image_size']),
            precision=metrics['precision'], recall=metrics['recall'],
            direct_ms=direct_ms, ransac_ms=ransac_ms, trials=info['trials']))
print(f'collected {len(rows)} rows across '
      f'{len({row["fraction"] for row in rows})} outlier fractions and '
      f'{len({row["seed"] for row in rows})} seeds')


In [ ]:
# TODO(student): summarise `rows` into the required table and plot.
# Report median direct/RANSAC corner RMSE, precision, recall, runtime, and trials
# per outlier fraction, then interpret the trend in your PDF.


## 7. B4: Coarse-to-fine translation study

B4 assumes `beta=0` and estimates translation only. It does not refine a general homography. Compare one and four levels, record the start and end estimate at every level, plot the RMSE history, and include a brightness-offset failure case.

The required sequence row uses the Oxford VGG `wall` pair `img1.ppm` → `img2.ppm` with `H1to2p`, and the 64 × 64 ROI `x=68:132, y=268:332` (end values excluded) in both frames. Start from `p0=(0,0)`. The reference translation is the displacement of the ROI center under `H1to2p`. Report `||p-p_reference||₂` and the corner RMSE between the four translated ROI corners and their homography-warped locations. Plot both frames with the patch, the one-level and four-level photometric histories, the reference and predicted quads, and both geometric errors.

The cell after the plotting TODO runs this experiment when the `wall` download is present. If it is missing, the cell runs a labeled synthetic `(8,-5)` smoke check so the notebook still executes. That smoke check earns no credit for the sequence row. Other sequences such as `graf` are not used for this row.

In [ ]:
# SUPPLIED: deterministic basin, sign, and brightness cases
template, image, true_p = make_translation_pair((8., -5.), seed=16722)
single_p, single_history = pa3.lucas_kanade_refine(template, image, np.zeros(2), 1)
pyramid_p, pyramid_history = pa3.lucas_kanade_refine(template, image, np.zeros(2), 4)
print('truth / one level / four levels:', true_p, single_p, pyramid_p)
print('8 full-resolution pixels at level 2:', 8 / 2**2)
print([(entry['level'], entry['p_full'], entry['status'])
       for entry in pyramid_history if entry['phase'] == 'level_end'])

In [ ]:
# SUPPLIED: worked example of one required plot, photometric RMSE by logged update.
iteration_rows = [entry for entry in pyramid_history if entry['phase'] == 'iteration']
plt.figure(figsize=(7,3))
plt.plot([entry['photometric_rmse'] for entry in iteration_rows])
plt.xlabel('logged update'); plt.ylabel('photometric RMSE'); plt.grid(True);
plt.savefig(ARTIFACTS / 'b4_lk_history.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# TODO(student): add the required error-by-pyramid-level plot.
# Each `level_end` record carries `level` and `p_full`. Plot the distance from
# `p_full` to `true_p` against level, for both the one-level and four-level runs.


In [ ]:
# SUPPLIED: Oxford wall experiment, with a labeled synthetic smoke check when offline
def _b4_gray(array):
    gray = np.asarray(array, dtype=np.float64)
    if gray.ndim == 3:
        gray = gray[..., :3] @ np.array([0.299, 0.587, 0.114])
    if gray.size and gray.max() > 1.0:
        gray = gray / 255.0
    return gray

def load_b4_sequence(data_root, prefer_vgg=True):
    sequence_root = Path(data_root) / 'vgg_affine' / 'wall'
    required = [sequence_root / 'img1.ppm', sequence_root / 'img2.ppm', sequence_root / 'H1to2p']
    if prefer_vgg and all(path.is_file() for path in required):
        frame1, frame2 = _b4_gray(plt.imread(required[0])), _b4_gray(plt.imread(required[1]))
        H_1_to_2 = np.loadtxt(required[2], dtype=np.float64)
        x0, y0, size = 68, 268, 64
        x1, y1 = x0 + size, y0 + size
        corners = np.array([[x0,y0],[x1-1,y0],[x1-1,y1-1],[x0,y1-1]], dtype=np.float64)
        target_corners, valid = project_points(H_1_to_2, corners)
        center = corners.mean(axis=0, keepdims=True)
        target_center, center_valid = project_points(H_1_to_2, center)
        if not valid.all() or not center_valid.all():
            raise RuntimeError('the fixed Oxford ROI has a non-finite homography projection')
        return dict(source='Oxford VGG wall: img1.ppm → img2.ppm', external=True,
            template=frame1[y0:y1,x0:x1], image=frame2[y0:y1,x0:x1],
            p_reference=target_center[0]-center[0], roi_corners=corners, target_corners=target_corners,
            display_source=frame1, display_destination=frame2, roi=(x0,y0,x1,y1))
    template, image, true_p = make_translation_pair((8.,-5.), seed=16722)
    height, width = template.shape
    corners = np.array([[0,0],[width-1,0],[width-1,height-1],[0,height-1]], dtype=np.float64)
    return dict(source='SYNTHETIC FALLBACK: shift (8,-5), seed 16722', external=False,
        template=template, image=image, p_reference=true_p, roi_corners=corners,
        target_corners=corners+true_p, display_source=template, display_destination=image,
        roi=(0,0,width,height))

def b4_sequence_metrics(study, p):
    predicted = study['roi_corners'] + np.asarray(p, dtype=np.float64)
    return dict(endpoint_error=float(np.linalg.norm(np.asarray(p)-study['p_reference'])),
        corner_rmse=float(np.sqrt(np.mean(np.sum((predicted-study['target_corners'])**2, axis=1)))))

def _closed(points):
    return np.vstack([points, points[0]])

def plot_b4_sequence(study, runs):
    fig, axes = plt.subplots(2, 2, figsize=(11,7))
    axes[0,0].imshow(study['display_source'], cmap='gray', vmin=0, vmax=1)
    axes[0,0].plot(*_closed(study['roi_corners']).T, color='yellow', lw=2)
    axes[0,0].set_title('frame 1 + fixed ROI')
    axes[0,1].imshow(study['display_destination'], cmap='gray', vmin=0, vmax=1)
    axes[0,1].plot(*_closed(study['target_corners']).T, color='lime', lw=2, label='homography/reference')
    colors = {1:'tab:red', 4:'tab:cyan'}
    for levels, run in runs.items():
        predicted = study['roi_corners'] + run['p']
        axes[0,1].plot(*_closed(predicted).T, '--', color=colors[levels], label=f'{levels}-level translation')
        iterations = [row for row in run['history'] if row['phase']=='iteration']
        axes[1,0].plot([row['photometric_rmse'] for row in iterations], color=colors[levels], label=f'{levels} level')
    axes[0,1].legend(fontsize=8); axes[0,1].set_title('frame 2: reference vs translated ROI')
    axes[1,0].set(xlabel='logged update', ylabel='photometric RMSE', title='LK histories')
    axes[1,0].grid(True); axes[1,0].legend()
    x = np.arange(2); width = .34
    for offset, (levels, run) in zip((-width/2,width/2), runs.items()):
        metrics = run['metrics']
        axes[1,1].bar(x+offset, [metrics['endpoint_error'], metrics['corner_rmse']], width,
            color=colors[levels], label=f'{levels} level')
    axes[1,1].set_xticks(x, ['endpoint','corner RMSE']); axes[1,1].set_ylabel('pixels')
    axes[1,1].set_title('geometric error'); axes[1,1].legend()
    for axis in axes[0]: axis.set_axis_off()
    fig.suptitle(study['source']); fig.tight_layout()
    fig.savefig(ARTIFACTS / 'b4_sequence.png', dpi=160, bbox_inches='tight')
    plt.show()

sequence_study = load_b4_sequence(DATA_ROOT)
if not sequence_study['external']:
    print('The wall sequence is not available, so this is the synthetic smoke check, not graded B4 evidence.')
    print('Rerun the data-setup cell near the top once you are online, then rerun this cell.')
sequence_runs = {}
for levels in (1,4):
    p, history = pa3.lucas_kanade_refine(sequence_study['template'], sequence_study['image'], np.zeros(2), levels)
    sequence_runs[levels] = dict(p=p, history=history, metrics=b4_sequence_metrics(sequence_study, p))
print(sequence_study['source'], '| ROI', sequence_study['roi'], '| reference', sequence_study['p_reference'])
print({levels: dict(p=run['p'], **run['metrics']) for levels, run in sequence_runs.items()})
plot_b4_sequence(sequence_study, sequence_runs)

## 8. C: SIFT and homography baseline

The supplied per-frame estimator uses up to 2000 SIFT features, ratio test 0.75, mutual matching, your RANSAC at 3 px and 0.99 confidence, and the supplied validity gates. You build the rest: the temporal state, the `STALE` display for at most three frames, whole-frame reinitialization, the clean and diagnostic views, and one improvement beyond this required baseline (see C2 in the PDF).

Record your own footage by default. The course-made clips in `data/fallback/` may replace only your own recording, and only with written staff approval given in advance for an access, privacy, safety, or accessibility barrier. Follow the README in that folder. Approval only replaces your own recording. Everything else in Parts C and D is still required.

In [ ]:
# SUPPLIED: fallback approval check. Setting this flag does not grant approval.
import json
import re
from data.generate_fallback import verify_fallback_package

USE_APPROVED_FALLBACK = False
FALLBACK_APPROVAL_ID = ''  # written staff response, format PA3-FB-YYYY-NNN
FALLBACK_ROOT = ROOT / 'data' / 'fallback'

if USE_APPROVED_FALLBACK:
    if re.fullmatch(r'PA3-FB-\d{4}-\d{3}', FALLBACK_APPROVAL_ID) is None:
        raise ValueError('enter the written staff approval ID before using fallback media')
    verify_fallback_package(FALLBACK_ROOT)
    fallback_ground_truth = json.loads((FALLBACK_ROOT / 'ground_truth.json').read_text())
    CAPTURE_TARGET_PATH = FALLBACK_ROOT / 'planar_target.png'
    CAPTURE_CLIPS = {name: FALLBACK_ROOT / f'{name}.mp4'
                     for name in ('baseline','fast_motion','occlusion','illumination')}
    print('APPROVED FALLBACK ACTIVE:', FALLBACK_APPROVAL_ID)
    print('Only your own recording is waived. All Part C and D analysis and artifacts are still required.')
else:
    fallback_ground_truth = None
    CAPTURE_TARGET_PATH = None  # TODO(student): set to your own planar-target image
    CAPTURE_CLIPS = {}          # TODO(student): add your own baseline and three variants
    print('OWN-CAPTURE PATH ACTIVE')

In [ ]:
# SUPPLIED: per-frame homography estimate. You build the temporal state around it.
def estimate_target_in_frame(target_rgb, frame_rgb, seed=SEED):
    src_xy, dst_xy, match_info = sift_correspondences(target_rgb, frame_rgb, max_features=2000, ratio=0.75)
    if len(src_xy) < 4:
        return None, dict(valid=False, reason='fewer than four matches')
    try:
        H, mask, ransac_info = pa3.ransac_homography(src_xy, dst_xy, 3.0, 0.99, np.random.default_rng(seed))
    except (ValueError, RuntimeError) as error:
        return None, dict(valid=False, reason=f'ransac_failed: {error}')
    errors = pa3.symmetric_transfer_error(H, src_xy, dst_xy)
    validity = tracking_validity(H, src_xy, mask, errors,
        target_size=(target_rgb.shape[1], target_rgb.shape[0]), frame_shape=frame_rgb.shape[:2], threshold_px=3.0)
    return H, dict(validity=validity, src_xy=src_xy, dst_xy=dst_xy, mask=mask,
                   errors=errors, ransac=ransac_info, matches=match_info)


### 8.1 Supplied smoke demonstration

The next cell runs the per-frame estimator and the supplied overlay on a few frames of a
course-made clip. Use it to check that your B2 and B3 code works on real video before you
record anything. This clip is for debugging only and earns no Part C or Part D credit. It
handles single frames only. You build the temporal state machine around it: tracking from the
last valid frame, the `STALE` display for at most three frames, hiding the overlay and
reinitializing afterwards, confidence, and your improvement.


In [ ]:
# SUPPLIED: single-frame smoke demonstration on a course-made debugging clip.
# Not Part C or D evidence. Replace with your own capture for anything you submit.
from common.pa3_utils import draw_planar_overlay, read_video

DEBUG_CLIP = ROOT / 'data' / 'fallback' / 'baseline.mp4'
debug_target = plt.imread(ROOT / 'data' / 'fallback' / 'planar_target.png')
debug_frames, debug_meta = read_video(DEBUG_CLIP, max_frames=4, frame_stride=25)
debug_target_size = (debug_target.shape[1], debug_target.shape[0])

figure, axes = plt.subplots(1, len(debug_frames), figsize=(4 * len(debug_frames), 3))
for axis, frame in zip(np.atleast_1d(axes), debug_frames):
    H_frame, diagnostics = estimate_target_in_frame(debug_target, frame)
    gates = diagnostics.get('validity')
    is_valid = bool(gates['valid']) if gates else False
    view = draw_planar_overlay(frame, H_frame, debug_target_size, valid=is_valid) \
        if H_frame is not None else frame
    title = f"valid={is_valid}"
    if gates:
        title += f" | inliers {gates['n_inliers']}/{gates['n_matches']} | conf {gates['confidence']:.2f}"
    axis.imshow(view); axis.set_title(title, fontsize=8); axis.set_axis_off()
figure.suptitle('SUPPLIED DEBUGGING CLIP (not your capture, not Part C or D evidence)')
figure.tight_layout()
figure.savefig(ARTIFACTS / 'c_debug_smoke.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# TODO(student): load your own capture and build the temporal system around the
# supplied per-frame estimator: last valid state, STALE display for at most three frames,
# hidden overlay with whole-frame reinitialization, scalar confidence, the clean
# application view, the synchronized diagnostic view, and your improvement.
# Keep raw private media outside the repository. Write only submission files to ARTIFACTS.


## 9. D: Controlled evidence and manual corners

Use your Part C clip as the baseline and record three more clips that each change one factor. Write your predictions before you evaluate. Mark the ordered target corners by hand on 5 to 10 frames per condition and report corner error or quad IoU, plus the valid-frame rate. Include the direct versus RANSAC comparison and the one-level versus four-level Lucas-Kanade comparison. D1 in the PDF explains how to choose the patch for that comparison.

In [ ]:
# TODO(student): replace with one ordered annotation and corresponding projected corners.
annotated_corners = np.array([[0.,0.],[99.,0.],[99.,79.],[0.,79.]])
predicted_corners = annotated_corners.copy()
manual_corner_metrics(predicted_corners, annotated_corners)

## 10. Hero Result and submission check

Export your Hero Result as an MP4 (H.264, yuv420p), a GIF, or one self-contained HTML file. It should run 10 to 30 seconds, at most 720p and 30 fps, and stay under 75 MB. It has to show:

- the original video and your result
- the diagnostic view, in sync with them
- one success, and one failure or recovery
- the same clip before and after your improvement
- what you assumed, and where the method breaks

Save one still image from it as well.

Upload both to Gradescope (**PA3 Code**). Posting the result in the Slack channel `#showcase-hw3` is optional and earns +1. Section 12 of the PDF has the steps.


In [ ]:
# TODO(student): after assembling RGB diagnostic_frames, export the final artifact.
# write_h264_video(ARTIFACTS / 'pa3_hero.mp4', diagnostic_frames, fps=30.0)
submission_audit = {
    'student_functions_complete': False, 'written_pdf_compiled': False,
    'manual_corner_metrics_included': False, 'hero_result_checked': False,
    'static_preview_included': False, 'resources_and_ai_disclosed': False,
}
submission_audit

In [ ]:
# SUPPLIED: make a backup ZIP of your code and artifacts. It does not replace the
# submission items in the PDF, so download those files individually too.
import zipfile
backup_path = ARTIFACTS / 'pa3_work_backup.zip'
with zipfile.ZipFile(backup_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(Path(pa3.__file__).parent.glob('*.py')):
        archive.write(path, 'student/' + path.name)
    for path in sorted(ARTIFACTS.rglob('*')):
        if path.is_file() and path.suffix.lower() != '.zip':
            archive.write(path, 'artifacts/' + path.relative_to(ARTIFACTS).as_posix())
print('Backup:', backup_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(backup_path))
print('Also download this executed notebook via File → Download → Download .ipynb.')
